# Padding

- 자연어 처리에서 각 문장(문서)의 길이는 서로 다를 수 있음
- 하지만 대부분의 모델은 고정 길이 입력을 기준으로 배치(batch) 단위 학습을 효율적으로 수행함
- 따라서 모든 문장의 길이를 동일한 길이(maxlen) 로 맞춰주는 작업이 필요함 → Padding

**Padding 개념**
- Padding: 짧은 문장에 PAD 같은 특수 토큰(보통 0) 을 채워 길이를 맞춤
- Truncation(잘라내기): 너무 긴 문장은 maxlen 기준으로 일정 길이까지만 남기고 자름
- Padding 방향
    - post padding: 뒤에 채움(일반적으로 많이 사용)
    - pre padding: 앞에 채움(모델/설정에 따라 사용)

**사용시기**
- 문장/문서를 시퀀스(정수 인덱스)로 바꾼 뒤 모델 입력으로 넣을 때
    - 예: Embedding + RNN/LSTM/GRU, 1D CNN, Transformer 계열 등
- 미니배치 학습(DataLoader/fit)에서 텐서 크기를 맞춰야 할 때
    - 배치로 묶으려면 (batch, seq_len) 형태로 길이가 동일해야 함
- 평가/추론에서도 동일하게 적용
    - 학습 때 사용한 maxlen 기준으로 테스트/서비스 입력도 동일 처리 필요

**코드 내 사용 위치**
- 텍스트 정제(소문자화/특수문자 처리 등)
- 토큰화(단어/서브워드)
- 정수 인코딩(Tokenizer, vocab 매핑)
- Padding/Truncation 적용
- 모델 입력(Embedding/Encoder) → 학습/평가

**Padding 이점**
- 일관된 입력 형식: 모든 문장이 동일한 길이의 시퀀스로 변환되어 모델 입력이 단순해짐
- 병렬 연산 최적화: 배치 단위 텐서 연산이 가능해져 GPU/행렬 연산 효율이 좋아짐
- 유연한 데이터 처리: 다양한 길이의 문서를 동일한 파이프라인으로 처리 가능

**주의사항(중요)**
- PAD는 의미 없는 값이므로 학습에 영향을 주면 안 됨
    - Masking(마스킹) 으로 PAD 위치를 손실 계산/어텐션 계산에서 제외하는 경우가 많음
- maxlen을 너무 크게 잡으면 PAD가 과도하게 늘어 연산 낭비 + 성능 저하가 생길 수 있음
    - 문장 길이 분포를 보고 적절한 maxlen을 선택하는 것이 좋음

In [2]:
preprocessed_sentences = [
    ['barber', 'person'],
    ['barber', 'good', 'person'],
    ['barber', 'huge', 'person'],
    ['knew', 'secret'],
    ['secret', 'kept', 'huge', 'secret'],
    ['huge', 'secret'],
    ['barber', 'kept', 'word'],
    ['barber', 'kept', 'word'],
    ['barber', 'kept', 'secret'],
    ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'],
    ['barber', 'went', 'huge', 'mountain']
]    # 문장별로 토큰화/정제된 결과를 리스트로 저장

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(preprocessed_sentences)
sequences = tokenizer.texts_to_sequences(preprocessed_sentences)
sequences

[[1, 5],
 [1, 8, 5],
 [1, 3, 5],
 [9, 2],
 [2, 4, 3, 2],
 [3, 2],
 [1, 4, 6],
 [1, 4, 6],
 [1, 4, 2],
 [7, 7, 3, 2, 10, 1, 11],
 [1, 12, 3, 13]]

In [5]:
tokenizer.word_index

{'barber': 1,
 'secret': 2,
 'huge': 3,
 'kept': 4,
 'person': 5,
 'word': 6,
 'keeping': 7,
 'good': 8,
 'knew': 9,
 'driving': 10,
 'crazy': 11,
 'went': 12,
 'mountain': 13}

In [6]:
tokenizer.index_word

{1: 'barber',
 2: 'secret',
 3: 'huge',
 4: 'kept',
 5: 'person',
 6: 'word',
 7: 'keeping',
 8: 'good',
 9: 'knew',
 10: 'driving',
 11: 'crazy',
 12: 'went',
 13: 'mountain'}

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded = pad_sequences(
    sequences,
    padding='post',
    maxlen = 3,
    truncating = 'post'
)

print(padded, padded.shape)

[[ 1  5  0]
 [ 1  8  5]
 [ 1  3  5]
 [ 9  2  0]
 [ 2  4  3]
 [ 3  2  0]
 [ 1  4  6]
 [ 1  4  6]
 [ 1  4  2]
 [ 7  7  3]
 [ 1 12  3]] (11, 3)


In [9]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.
The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.
On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.
Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.
The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""

sentences = sent_tokenize(raw_text)

en_stopwords = stopwords.words('english')

vocab = {}

preprocessed_sentences = []

for sentence in sentences:
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in en_stopwords]
    tokens = [token for token in tokens if len(token) > 2]

    for token in tokens:
        if token not in vocab:
            vocab[token] = 1
        else:
            vocab[token] += 1

    preprocessed_sentences.append(tokens)

print(preprocessed_sentences)

[['little', 'prince', 'written', 'antoine', 'saint-exupéry', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth'], ['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes'], ['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince'], ['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'b-612', 'lives', 'alone', 'rose', 'loves', 'deeply'], ['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets'], ['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox'], ['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior'], ['earth', 'little', 'prince', 'meets', 'various', 'creatures', 'including', 'fox', 'teaches', 'relationships', 'importance', 'taming', 'means', 'building', 'ties', 'others'], ['fox', 'famous', 'line', 'become', 'responsible', 'forever', 'tamed', 'resonates', '

In [10]:
# 상위 15개만 사용(필터링용), 그 외 토큰은 OOV 처리
tokenizer = Tokenizer(num_words=15, oov_token='<OOV>')

# 단어 빈도 기반 인덱스 사전
tokenizer.fit_on_texts(preprocessed_sentences)
sequences = tokenizer.texts_to_sequences(preprocessed_sentences)
sequences

[[3, 2, 1, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9],
 [10, 1, 4, 1, 1, 1, 11, 1],
 [1, 1, 11, 12, 1, 7, 1, 3, 2],
 [3, 2, 1, 1, 13, 1, 1, 1, 1, 5, 1, 1],
 [1, 1, 4, 1, 1, 1, 1],
 [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6],
 [1, 2, 1, 1, 14, 1, 1, 1, 1, 1],
 [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1],
 [6, 1, 1, 1, 1, 1, 1, 1, 2, 1, 5],
 [1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 4, 1, 1, 13, 1, 5],
 [10, 1, 4, 1, 14, 1, 3, 2, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

In [13]:
padded = pad_sequences(sequences)

print(padded, padded.shape)

[[ 0  0  0  3  2  1  1  1  1  1  7  2  1  1  8  9]
 [ 0  0  0  0  0  0  0  0 10  1  4  1  1  1 11  1]
 [ 0  0  0  0  0  0  0  1  1 11 12  1  7  1  3  2]
 [ 0  0  0  0  3  2  1  1 13  1  1  1  1  5  1  1]
 [ 0  0  0  0  0  0  0  0  0  1  1  4  1  1  1  1]
 [ 0  0  0  0  0  8  1  1  1  1  1  1  1  1  1  6]
 [ 0  0  0  0  0  0  1  2  1  1 14  1  1  1  1  1]
 [ 9  3  2 12  1  1  1  6  1  1  1  1  1  1  1  1]
 [ 0  0  0  0  0  6  1  1  1  1  1  1  1  2  1  5]
 [ 0  0  0  0  0  0  1  3  2  1  1  1  1  1  1  1]
 [ 0  0  0  0  0  0  0  0  1  1  4  1  1 13  1  5]
 [ 0  0  0  0  0 10  1  4  1 14  1  3  2  1  1  1]
 [ 0  0  0  1  1  1  1  1  1  1  1  1  1  1  1  1]] (13, 16)
